# Allpass FDN completion

For a given feedback matrix **A**, the goal is to construct **b**, **c**, and **d** such that the FDN is uniallpass.

See *Allpass Feedback Delay Networks*, Sebastian J. Schlecht (IEEE Trans. Signal Processing).

— Original MATLAB: Sebastian J. Schlecht, 9 June 2020

## Setup

In [ ]:
import numpy as np
import pyFDN

np.random.seed(2)

## Test: complete orthogonal

Given **A** from a random orthogonal matrix, complete to **V = [A,b;c,d]** orthogonal.

In [ ]:
N = 3
num_io = 2
V = pyFDN.random_orthogonal(N + num_io)
A = V[:N, :N]

b, c, d, VV = pyFDN.complete_orthogonal(A, num_io)
assert pyFDN.is_orthogonal(VV, tol=1e-6), "VV should be orthogonal"
print("Test complete orthogonal: OK")

## Test: complete diagonally similar to orthogonal

**A** is diagonally similar to an orthogonal block; complete to uniallpass.

In [ ]:
N = 3
num_io = 1
X1 = np.block([[np.diag(np.random.rand(N)), np.zeros((N, 1))], [np.zeros((1, N)), np.array([[1.0]])]])
V = pyFDN.random_orthogonal(N + num_io)
XVX = np.linalg.solve(X1, V @ X1)
XAX = XVX[:N, :N]

b, c, d, X, V_bal = pyFDN.complete_allpass_fdn(XAX)
assert pyFDN.is_orthogonal(V_bal, tol=1e-6), "V should be orthogonal"
print("Test complete diagonally similar to orthogonal: OK")

## Test: complete series allpass

**A** from Schroeder series allpass; complete to uniallpass.

In [ ]:
N = 4
g = np.random.uniform(0.5, 0.99, N)

A, b, c, d = pyFDN.series_allpass(g)

b, c, d, X, V_bal = pyFDN.complete_allpass_fdn(A)
assert pyFDN.is_orthogonal(V_bal, tol=1e-6), "V should be orthogonal"
print("Test complete series allpass: OK")

## Test: nested allpass

**A** from nested allpass; complete to uniallpass. (Sometimes fails due to poor low-rank solution.)

In [ ]:
N = 3
g = np.random.uniform(0.8, 0.9, N)
A, b, c, d = pyFDN.nested_allpass(g)

b, c, d, X, V_bal = pyFDN.complete_allpass_fdn(A)
# assert pyFDN.is_orthogonal(V_bal, tol=1e-6), "V should be orthogonal"
# print("Test nested allpass: OK")
# TODO: the completion fails for this example

## Test: homogeneous allpass

**A** from homogeneous allpass FDN; verify uniallpass then complete again.

In [ ]:
delays = np.array([32, 19, 13])
g = 0.99
G = np.diag(g ** delays)
X_diag = -np.diag([0.4, 0.6, 0.85])
A, b, c, d = pyFDN.homogeneous_allpass_fdn(G, X_diag)[:4]
is_a, X_lyap = pyFDN.is_uniallpass(A, b, c, d)

b, c, d, X, V_bal = pyFDN.complete_allpass_fdn(A)
assert pyFDN.is_orthogonal(V_bal, tol=1e-6), "V should be orthogonal"
print("Test homogeneous allpass: OK")
print("\nAll completion tests passed.")